In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import rioxarray as rxr
import cartopy.crs as ccrs

In [3]:
from pathlib import Path

rsut_path = Path("../../data/CanESM5_1850-2100_rsut.nc")
rsutcs_path = Path("../../data/CanESM_1850-2100_rsutcs.nc")
output_path = Path("../../data/CanESM_1850-2100_rsutcre.nc")

rsut_ds = xr.open_dataset(rsut_path, engine="netcdf4")
rsutcs_ds = xr.open_dataset(rsutcs_path, engine="netcdf4")


In [4]:

rsut_var = "rsut" if "rsut" in rsut_ds.data_vars else list(rsut_ds.data_vars)[0]
rsutcs_var = "rsutcs" if "rsutcs" in rsutcs_ds.data_vars else list(rsutcs_ds.data_vars)[0]

rsut_da = rsut_ds[rsut_var]
rsutcs_da = rsutcs_ds[rsutcs_var]

rsut_da, rsutcs_da = xr.align(rsut_da, rsutcs_da, join="inner")

ESM5_data = xr.merge([
    rsut_da.to_dataset(name="rsut"),
    rsutcs_da.to_dataset(name="rsutcs"),
])

swcre = ESM5_data["rsut"] - ESM5_data["rsutcs"]

if "member" in swcre.dims:
    swcre = swcre.mean("member")

swcre_ds = swcre.to_dataset(name="cre")
swcre_ds["cre"].attrs["long_name"] = "Shortwave Cloud Radiative Effect"
swcre_ds["cre"].attrs["description"] = "Computed as rsut - rsutcs"

swcre_ds.to_netcdf(output_path)

print(f"Built CRE dataset: {output_path}")
print(swcre_ds)

Built CRE dataset: ..\..\data\CanESM_1850-2100_rsutcre.nc
<xarray.Dataset> Size: 99MB
Dimensions:  (time: 3012, lat: 64, lon: 128)
Coordinates:
  * time     (time) object 24kB 1850-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat      (lat) float64 512B -87.86 -85.1 -82.31 -79.53 ... 82.31 85.1 87.86
  * lon      (lon) float64 1kB 0.0 2.812 5.625 8.438 ... 348.8 351.6 354.4 357.2
Data variables:
    cre      (time, lat, lon) float32 99MB 9.912 9.971 10.03 ... 0.0 0.0 0.0
